## 2. `RegenCascade.ipynb`
 
### Purpose
Drives the regeneration algorithm across an entire system `f1, f2, …, fn`
one function ("codimension") at a time. At each stage it needs a set of
**start points** for the homotopy that adds the next equation; the
`BasePointRegen` class builds those start points for the very first stage,
and `SliceMover` (from the other notebook) is reused to advance the slice
between stages.

In [2]:
%run SliceMover.ipynb
import bertini as b2
import numpy as np
from bertini import nag_algorithm
from bertini import Slice

### Class `BasePointRegen`
 
Stores the system, its first variable group (`self.vars = system.variable_groups()[0]` — assuming a single variable group), and
`self.dim = system.num_functions()`. Several result attributes
(`base_points_numeric`, `start_moving_system`, `end_moving_system`,
`degree`, `start_points`) start as `None` and are filled in by
`generate_start_points`.
 
#### `generate_start_points(bottom_slice, *, rng=None)`
 
Builds the base-level (codimension-1) start points for regeneration by
using the product of linear forms intersected with a
generic linear slice, which can be solved directly, without a homotopy.
 
Step by step:
 
1. **Degree lookup.** `degree = list(self.system.degrees())[0]` — the
   degree of the system's first function, `f1`. 
2. **Randomize to codimension 1.**
   `self.randomized_sys = self.system.randomize(codimension=1)` collapses
   the system to a single representative equation.
3. **Build two parallel container systems**, `start_moving` and
   `end_moving`, both starting from an empty system carrying only the
   variable group:
   - `start_moving` gets the product of `degree`
     random linear forms (`add_products_of_linears`), each factor's
     coefficients drawn via `b2.random_complex()` and converted to
     Bertini's symbolic complex type.
   - `end_moving` gets the real (randomized) target function
     `target_fn`.
   - `bottom_slice.add_to(...)` appends the same generic linear slice
     equations to both systems, cutting each down to isolated points.
4. **Solve the start system in closed form.** Since `start_moving`'s
   defining equation is a product of `degree` linear factors, its
   intersection with the slice is just `degree` separate linear systems. 
   For each factor's coefficient row, the code stacks
   that row with the slice's coefficient rows into a square matrix `A`
   and solves `A·x = b` with `np.linalg.solve` — giving the `degree` base
   points directly, with no path tracking needed.
5. **Convert to Bertini points** via `complex_mp`, and store:
   `self.start_points` (Bertini form), `self.base_points_numeric` (raw
   NumPy), `self.start_moving_system`, `self.end_moving_system`,
   `self.degree`.

In [3]:
class BasePointRegen:
    def __init__(self, system):
        self.system = system
        self.randomized_sys = None
        self.vars = system.variable_groups()[0]
        self.dim = system.num_functions()

        self.start_points = None   # base-level start points, set by generate_start_points

        self.base_points_numeric = None
        self.start_moving_system = None
        self.end_moving_system = None
        self.degree = None

    def generate_start_points(self, bottom_slice, *, rng=None):
        #rng setup
        if rng is None:
            rng = np.random.default_rng()

        #var setup
        degree = list(self.system.degrees())[0]
        
        #randomizes R
        self.randomized_sys = self.system.randomize(codimension = 1)

        target_fn = self.randomized_sys.function(0)
        bottom_slice_coeffs = bottom_slice.coefficients()

        #systems setup
        fixed_sys = b2.System()
        fixed_sys.add_variable_group(self.vars)

        start_moving = b2.system.clone(fixed_sys)
        end_moving = b2.system.clone(fixed_sys)

        #random coeff for the degree
        factor_coeffs = [[b2.random_complex() for ii in range(self.dim + 1)] for jj in range(degree)]

        #converts into symbolics complex numbers
        def convert_to_b2_symbolic_complex(z):
            return b2.symbolics.Complex(str(z.real), str(z.imag))

        #add functions to the systems
        start_moving.add_products_of_linears([[[convert_to_b2_symbolic_complex(c) for c in row] for row in factor_coeffs]])
        end_moving.add_function(target_fn)

        bottom_slice.add_to(start_moving)
        bottom_slice.add_to(end_moving)

        #matrix setup
        base_points_numeric = []
        for row in factor_coeffs:
            row = np.array(row)
            A = np.vstack([row[:-1][None, :], bottom_slice_coeffs[:, :-1]])       # N x N
            b = -np.concatenate([[row[-1]], bottom_slice_coeffs[:, -1]])            # N

            base_points_numeric.append(np.linalg.solve(A.astype(complex), b.astype(complex)))

        #converts points into a numpy array of complex_mp points
        def to_bertini_point(pt):
            return np.array([b2.complex_mp(str(v.real), str(v.imag)) for v in pt])

        base_points_bertini_form = [to_bertini_point(p) for p in base_points_numeric]

        #results
        self.start_points = base_points_bertini_form
        self.base_points_numeric = base_points_numeric
        self.start_moving_system = start_moving
        self.end_moving_system = end_moving
        self.degree = degree

### Driver cell
 
Builds a concrete 3-variable example and runs the codimension-by-codimension loop:
 
```python
x, y, z = b2.Variable('x'), b2.Variable('y'), b2.Variable('z')
f1 = (y - x**2) * (x**2 + y**2 + z**2 - 1) * (x - 2)
f2 = (z - x**3) * (x**2 + y**2 + z**2 - 1) * (y - 2)
f3 = (z - x**3) * (y - x**2) * (x**2 + y**2 + z**2 - 1) * (z - 2)
```
`sys` gets the variable group `[x, y, z]` and all three functions;
`bpr = BasePointRegen(sys)`, and a maximal generic slice
`bottom_slice = b2.Slice.random_complex(sys.variable_groups(), sys.num_variables() - 1)`
is built once, up front (2 linear equations, since there are 3 variables).
 
The main loop, `for codim in range(1, bpr.dim + 1)` (i.e. codim 1, 2, 3):
 
- **`codim == 1`:**
  Calls `bpr.generate_start_points(bottom_slice)`, pulls out
  `start_points`, `start_system`, `target_system`, and — for
  diagnostics — evaluates `start_system` at every start point and prints
  the residual. An inline comment flags an important numerical caveat:
  **raw residual magnitude isn't scale-invariant**, so a fixed tolerance
  like `1e-13` can't reliably tell a true root from a non-root once the
  system has been rescaled (e.g. by randomization).
- **`codim > 1`:**
  Takes the slices from the previous stage's target system
  (`target_system.slices()`), drops the first one
  (`target_slices = slices_prev_step[1:]`), and builds a fresh random
  2-dimensional slice (`Slice.random_complex(vars, 2)`). A new
  `SliceMover(bpr.randomized_sys, slices_prev_step[0], rand_slice, start_points)` is created, then `sm.move_slice()` is called in a loop that runs `bpr.degree` times, each time collecting `results.solutions` into
  `new_start_points` and advancing to a fresh random end slice via `sm.set_end_slice(...)`. 
- **Every iteration (both branches):**
  Builds the actual regeneration homotopy that adds the next equation:
```python
  hom = b2.nag_algorithm.blend_homotopy(target=target_system, start=start_system)
  homSolver = b2.HomotopySolver(target=target_system, homotopy=hom, start_points=start_points)
  solve_result = homSolver.solve()
```
  `solve_result` isn't consumed further in the code shown (its print
  statement, and a following call to a not-yet-implemented
  `bpr.set_up_partially_regen_system(codim)`, are both commented out).
 
---
 
#### Known issues and loose ends
 
1. **"Junk removal"** after the repeated `move_slice()` calls in the
   `codim > 1` branch is flagged as not yet implemented.


In [4]:
#driver

#function setup
x, y, z = b2.Variable('x'), b2.Variable('y'), b2.Variable('z')
#assuming these are in non-increasing degree order (expose sort so that users can have the system automaticaly sorted)
f1 = (y - x**2) * (x**2 + y**2 + z**2 - 1) * (x - 2)
f2 = (z - x**3) * (x**2 + y**2 + z**2 - 1) * (y - 2)
f3 = (z - x**3) * (y - x**2) * (x**2 + y**2 + z**2 - 1) * (z - 2)

#systems setup
sys = b2.System()
vars = b2.VariableGroup([x, y, z])
sys.add_variable_group(vars)
clone_sys = sys.clone() #used to copy the base system without the added functions if needed later
sys.add_functions([f1, f2, f3])
#sys.sort()

bpr = BasePointRegen(sys)

bottom_slice = b2.Slice.random_complex(sys.variable_groups(), sys.num_variables() - 1)

for codim in range(1, bpr.dim + 1):
    print(f'codim {codim}')
    if codim==1:
        result = bpr.generate_start_points(bottom_slice)
        # print(f"degree(f{codim + 1}) = {bpr.degree}  ->  {len(bpr.start_points)} base-level start points")
        # for p in bpr.base_points_numeric:
        #     print(" ", p)
            
        # print(bpr.start_moving_system)
        # print(bpr.end_moving_system)

        start_points = bpr.start_points
        start_system = bpr.start_moving_system
        target_system = bpr.end_moving_system

        # residual cannot be used to determine if a point is a solution or not based on scale
        # if x is a solution where x < 10^-11 or similarly close enough to zero where we would assume it is
        # then A * f(x), where A is 10^11 or larger makes x no longer a solution that is close enough to 0,
        # but instead 1. 
        # for p in bpr.start_points:
        #     print("eval:\n", *start_system.eval(p)) 

        # print("\n\n")

    else:
        # use the endpoints from the previous codim's solve
        # start_system = clone_sys.clone()
        # f1 and product of linears from slicemove
        # code here that takes the linear product of the slices we moved to in previous line
        slices_prev_step = target_system.slices()
        #print("target_sys",target_system)

        target_slices = slices_prev_step[1:] # take the top slice off and replace with the real system

        #print(f"len(slices_prev_step): {len(slices_prev_step)}\nlen(target_slices):{len(target_slices)}")

        new_start_points = []
        print("rand_sys:", bpr.randomized_sys)
        print("slices_prev_step:",slices_prev_step[0])
        rand_slice = Slice.random_complex(vars, 2)
        print("rand_slice:",rand_slice)

        sm = SliceMover(bpr.randomized_sys, slices_prev_step[0], rand_slice, start_points)
        
        for ii in range(bpr.degree):
            print("hom: ", sm.hom)
            results = sm.move_slice()
            print("results = ", results)
            #junk removal here
            new_start_points.append(results.solutions)
            sm.set_end_slice(Slice.random_complex(vars, 2))

        print("new start points: ", *new_start_points)


    # do the homotopy solve to solve the first equation in the system
    print("start_sys:\n", *bpr.start_points)

    hom = b2.nag_algorithm.blend_homotopy(target = target_system, start= start_system)
    homSolver = b2.HomotopySolver(target=target_system, homotopy=hom, start_points=start_points)
    solve_result = homSolver.solve()

    #print(solve_result)

    #when checking if == 0, use abs(x) >  1 * 10^-13
    #bpr.set_up_partially_regen_system(codim)

codim 1
start_sys:
 [(-1.0421075930853945000020e+00, 1.3575679903465262999980e+00)
 (1.1196502975482482999980e+00, 1.2146505795945861000000e-01)
 (-4.5905432682353849999980e-01, -5.8915688836732570000050e-01)] [(1.2066938925426353000010e-01, 4.6983945322518605000070e-01)
 (-2.5258262307363083000070e-01, 1.3030530841713234999970e+00)
 (5.3674869763359160000100e-01, -6.8625177450125760000060e-01)] [(-2.2410193566864312000040e-01, 1.5548508551481523999990e+00)
 (9.4485329646279010000100e-02, -6.2783987296640339999920e-02)
 (-5.4613936738801480000000e-02, -1.7973703369834137999970e-01)] [(-1.9574953655444255999970e+00, 3.4344881486158657000020e-01)
 (2.3245977432390080000050e+00, 1.3079889924555659999980e+00)
 (-6.2579092806664770000040e-01, -1.5085079986100937999980e+00)] [(-3.9466412111166010000020e-02, 4.7342747385981593000080e-01)
 (-5.4964425796656279999910e-02, 1.2869650186154873000010e+00)
 (4.4236570472771129999990e-01, -7.4186445690234450000060e-01)]
codim 2
rand_sys: 1 variable g